In [9]:
# Import necessary libraries
import json
import bisect
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
# Get the JSON objects representing the papers
path = Path("../data/cleaned_papers_text.json")
with open(path, 'r', encoding='utf-8') as f:
    papers = json.load(f)

print(f"Retrieved {len(papers)} papers")

Retrieved 10 papers


In [11]:
# Create the array of Documents
SEP = '\n'
documents = []
page_tables = {}

for paper in papers:
    texts = [page['text'].strip() for page in paper['pages']]
    nums = [page['page_number'] for page in paper['pages']]

    starts, pos = [], 0
    for t in texts:
        starts.append(pos)
        pos += len(t) + len(SEP)

    page_tables[paper['id']] = (starts, nums)

    documents.append(Document(
        page_content=SEP.join(texts),
        metadata={
            'paper_id': paper['id'],
            'title': paper['title'],
            'category': paper['category'],
            'pdf_url': paper['pdf_url'],
            'total_pages': paper['total_pages']
        }
    ))

print(f"Created {len(documents)} documents")

Created 10 documents


In [12]:
# Recursive split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=125,
    add_start_index=True
)
chunks = splitter.split_documents(documents)

for c in chunks:
    starts, nums = page_tables[c.metadata['paper_id']]
    si = c.metadata['start_index']
    ei = si + len(c.page_content) - 1

    i = bisect.bisect_right(starts, si) - 1
    j = bisect.bisect_right(starts, ei) - 1

    c.metadata['page_number'] = nums[i]
    c.metadata['end_page'] = nums[j]

total_chunks = len(chunks)
avg_chunk_size = sum(len(c.page_content) for c in chunks) / total_chunks
print(f"Total chunks: {total_chunks}")
print(f"Average chunk size: {avg_chunk_size:.1f}")
print("First chunk:")
print(chunks[0].page_content)

Total chunks: 918
Average chunk size: 954.0
First chunk:
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis , Ethan Perez ,
Aleksandra Piktus , Fabio Petroni , Vladimir Karpukhin , Naman Goyal , Heinrich Kuttler ,
Mike Lewis , Wen-tau Yih , Tim Rocktaschel , Sebastian Riedel , Douwe Kiela 
 Facebook AI Research; University College London; New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when fine-tuned on downstream NLP tasks. However, their ability to access and precisely manipulate knowledge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-specific architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pretrained models with a differentiable access mechanism to explicit non-parametric
memory have so far be

In [13]:
# Write chunks to JSON file, Tanish uses these to store in Chroma DB
json_chunks = []
for c in chunks:
    json_chunks.append(
        {
            'page_content': c.page_content,
            'metadata': c.metadata
        }
    )

out_path = Path("../data/chunks.json")
with open(out_path, 'w', encoding='utf-8') as file:
    json.dump(json_chunks, file, ensure_ascii=False, indent=2)